# Multilevel Monte Carlo for Option Pricing

Demonstrates MLMC for Asian option pricing using `FinancialOptionML` with geometric Brownian motion paths at multiple resolution levels.

In [1]:
using QMC
import QMC: Uniform

## Setup

GBM with 16 time steps at the finest level, coarsest level at 4 steps.

In [2]:
# GBM path model
dd = IIDStdUniform(16; seed=7)
tm = GeometricBrownianMotion(dd;
    volatility=0.2,
    start_price=100.0,
    interest_rate=0.05,
    t_final=1.0)

# Multilevel integrand
fml = FinancialOptionML(tm; d_coarsest=4)
println("Levels:")
for l in 0:2
    println("  l=$l: $(dimension_at_level(fml, l)) time steps")
end

Levels:
  l=0: 4 time steps
  l=1: 8 time steps
  l=2: 16 time steps


## Exact Reference Value

In [3]:
# Geometric Asian call with Black-Scholes formula
fo_exact = FinancialOption(tm; option_type=:asian, strike_price=100.0,
                            asian_mean=:geometric)
exact = get_exact_value(fo_exact)
println("Exact geometric Asian option price: $(round(exact, digits=4))")

Exact geometric Asian option price: 5.8417


## Per-Level Variance Decay

In [4]:
using Statistics

let
    for l in 0:2
        d_l = dimension_at_level(fml, l)
        x_level = randn(1000, d_l)
        Pc, Pf = ml_evaluate(fml, x_level, l)
        dP = Pf .- Pc
        println("Level $l: E[Pf-Pc]=$(round(mean(dP), digits=5)), " *
                "Var[Pf-Pc]=$(round(var(dP), digits=5)), " *
                "E[Pf]=$(round(mean(Pf), digits=4))")
    end
end

Level 0: E[Pf-Pc]=0.0, Var[Pf-Pc]=0.0, E[Pf]=0.0
Level 1: E[Pf-Pc]=0.0, Var[Pf-Pc]=0.0, E[Pf]=0.0
Level 2: E[Pf-Pc]=0.0, Var[Pf-Pc]=0.0, E[Pf]=0.0
